In [ ]:
import ROOT, glob, os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams['figure.dpi'] = 400
plt.rcParams['savefig.dpi'] = 400

In [ ]:
dirDir= "/home/users/l/lebrun/"
resultPath = dirDir + "results/"
figPath = dirDir + "figures/"
dataPath = "/dpnc/beegfs/dampe/tryf/ForKimberly/"
exposurePath = "/dpnc/beegfs/dampe/tryf/ForKimberly/exposureMaps_GIE/"
exponent = 0

In [ ]:
energy_bin = 5
# Load the table once globally
PATH_EFFAREA = "/home/users/l/lebrun/results/effArea/effectiveArea_weighted.npz"
PATH_GIEM_BINS = "/home/users/l/lebrun/results/all_bins_GIEM.npz"
# Load the effective area table
EFF_AREA_NPZ = np.load(PATH_EFFAREA)
EFF_AREA_TABLE = EFF_AREA_NPZ['effArea'] # Shape (n_E_bins, n_costheta_bins, n_phi_bins) ~ (20, 10, 360)

BINS_NPZ = np.load(PATH_GIEM_BINS)

# Get bins edges and centers
EAREA_E_BINS_EDGES = EFF_AREA_NPZ['E_bins_edge']
EAREA_E_BINS_CENTERS = np.sqrt(EAREA_E_BINS_EDGES[:-1] * EAREA_E_BINS_EDGES[1:])  # GeV

ENERGY_BINS_EDGES = BINS_NPZ['E_bins_edge']
ENERGY_BINS_CENTERS = np.sqrt(ENERGY_BINS_EDGES[:-1] * ENERGY_BINS_EDGES[1:])  # GeV
COSTHETA_BINS_EDGES = EFF_AREA_NPZ['theta_bins_edge']
PHI_BINS_EDGES = EFF_AREA_NPZ['phi_bins_edge']

In [ ]:
energy = ENERGY_BINS_CENTERS[energy_bin]

In [ ]:
dataDir = "/home/users/l/lebrun/data/"
exposure_file = np.load(resultPath + f"exposureMaps/{energy:.3f}GeV_exposure_sum.npz")

lon_edges = exposure_file['lon_edges']
lat_edges = exposure_file['lat_edges']
exposure_map = exposure_file['exposure_map']

print(f"DEBUG : lons : {lon_edges[0]} - {lon_edges[-1]}, lat : {lat_edges[0]}-{lat_edges[-1]}, exposure_map : {exposure_map}")

dataName = "DmlNtup_Data_9years_FullPhotonData_AllSky.root"
tree = None
f = ROOT.TFile(dataDir + dataName, "READ") 

tree = f.Get("DmlNtup")
print(tree)
print(type(tree))
rdf = ROOT.RDataFrame(tree)
df = rdf.AsNumpy(["celestialcoordBest_2", "celestialcoordBest_3", "tt_bgoTotalEcorr_GeV"])
df = pd.DataFrame(df)

df = df.rename(columns={"celestialcoordBest_2" : "lon", "celestialcoordBest_3" : "lat", 
                        "tt_bgoTotalEcorr_GeV" : "energy"})

edge_up = np.digitize(energy, ENERGY_BINS_EDGES) 
edge_low = edge_up - 1

energy_up = ENERGY_BINS_EDGES[edge_up]
energy_low = ENERGY_BINS_EDGES[edge_low]

print(f"DEBUG : edges indices : {edge_up}, {edge_low}")
print(f"DEBUG : energy = {energy}")
print(f"DEBUG : energy edges : {ENERGY_BINS_EDGES[edge_low]}, {ENERGY_BINS_EDGES[edge_up]}")
print(f"DEBUG : all centers : {ENERGY_BINS_CENTERS}")

In [ ]:
def conv_lon_rad(target_lon):
    target_lon = np.where(target_lon > np.pi, target_lon - 2*np.pi, target_lon)
    return target_lon
print(f"DEBUG : df['lon'] : {df['lon'].min()} - {df['lon'].max()} ; df['lat'] : {df['lat'].min()} - {df['lat'].max()}")
df['lon'] = conv_lon_rad(df['lon'])
print(f"DEBUG : df['lon'] : {df['lon'].min()} - {df['lon'].max()} ; df['lat'] : {df['lat'].min()} - {df['lat'].max()}")

In [ ]:
df_ene = df.loc[(df['energy'] < energy_up) & (df['energy'] > energy_low)]
print(f"DEBUG : len(df) = {len(df)}, len(df_ene) = {len(df_ene)}")

In [ ]:
counts_grid_ene, _, _ = np.histogram2d(
    df_ene["lon"], df_ene["lat"], 
    bins=[np.sort(lon_edges), lat_edges], 
)
counts_grid, _, _ = np.histogram2d(
    df["lon"], df["lat"], 
    bins=[np.sort(lon_edges), lat_edges], 
)

In [ ]:
# Création du plot en projection Mollweide
import matplotlib.colors as mcolors
fig1 = plt.figure(figsize=(8, 5))
ax1 = fig1.add_subplot(111, projection="mollweide")
ax1.set_facecolor('black')
# Plot the 2D histogram using pcolormesh
# We use the logarithmic normalization for the color scale
cmap = "plasma"
norm = mcolors.LogNorm()  # Use logarithmic normalization for color scale
pc = ax1.pcolormesh(lon_edges, lat_edges, counts_grid.T, cmap=cmap, norm=norm, shading='auto')

# Colorbar
cbar = fig1.colorbar(pc, ax=ax1, orientation="horizontal", fraction=0.05, pad=0.07)
cbar.set_label(r"Counts")

# Modifying axis labels and grid
ax1.xaxis.set_tick_params(labelcolor="white")
tick_labels = np.array([150, 120, 90, 60, 30, 0, -30, -60, -90, -120, -150])
ax1.set_xticklabels(tick_labels)
ax1.set_ylabel(r"$b$ [deg]")
# ax1.grid(True, linestyle="-", alpha=0.5)
# Show the plot
plt.show()

In [ ]:
fig1 = plt.figure(figsize=(8, 5))
ax1 = fig1.add_subplot(111, projection="mollweide")
ax1.set_facecolor('black')
# Plot the 2D histogram using pcolormesh
# We use the logarithmic normalization for the color scale
cmap = "viridis"
norm = mcolors.LogNorm()  # Use logarithmic normalization for color scale
pc = ax1.pcolormesh(lon_edges, lat_edges, exposure_map.T, cmap=cmap, norm=norm, shading='auto')

# Colorbar
cbar = fig1.colorbar(pc, ax=ax1, orientation="horizontal", fraction=0.05, pad=0.07)
#cbar.set_label(r"Exposure (s$^{-1}$cm$^{-2}$sr$^{-1}$)")
cbar.set_label(r"Exposure (cm$^{2}$s)")

# Modifying axis labels and grid
ax1.xaxis.set_tick_params(labelcolor="white")
tick_labels = np.array([150, 120, 90, 60, 30, 0, -30, -60, -90, -120, -150])
ax1.set_xticklabels(tick_labels)
ax1.set_ylabel(r"$b$ [deg]")
# ax1.grid(True, linestyle="-", alpha=0.5)
# Show the plot
plt.show()

In [ ]:
flux_map = np.divide(counts_grid_ene, exposure_map, where=(exposure_map > 1e-16))


In [ ]:
print(flux_map.shape)

In [ ]:
fig1 = plt.figure(figsize=(8, 5))
ax1 = fig1.add_subplot(111, projection="mollweide")
ax1.set_facecolor('black')
# Plot the 2D histogram using pcolormesh
# We use the logarithmic normalization for the color scale
cmap = "plasma"
norm = mcolors.LogNorm()  # Use logarithmic normalization for color scale
pc = ax1.pcolormesh(lon_edges, lat_edges, flux_map.T, cmap=cmap, norm=norm, shading='auto')

# Colorbar
cbar = fig1.colorbar(pc, ax=ax1, orientation="horizontal", fraction=0.05, pad=0.07)
cbar.set_label(fr"Flux (ph s$^{{-1}}$cm$^{{-2}}$)")
#cbar.set_label(r"Flux (cm$^{2}$\,s)")

# Modifying axis labels and grid
ax1.xaxis.set_tick_params(labelcolor="white")
tick_labels = np.array([150, 120, 90, 60, 30, 0, -30, -60, -90, -120, -150])
ax1.set_xticklabels(tick_labels)
ax1.set_ylabel(r"$b$ [deg]")
# ax1.grid(True, linestyle="-", alpha=0.5)
# Show the plot
fig1.savefig(figPath+f"fluxMaps/bin_{energy_bin}.png")
plt.show()